# GPU Hello World

### Task 3

1. Make a NumPy matrix
2. Install, import and initialize PyCUDA
3. Make a GPU matrix
4. Copy the NumPy matrix data to the GPU
5. Write a GPU *kernel* that does some kind of computation on an input matrix
6. Copy the result back to the CPU
7. Print and plot both matrices

### Solution 3

<-- see `hello_world_pycuda_solution.ipynb`

In [1]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 20.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.0 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=19d9c4b23a7a18402ad8410018479ce65d4d1429551fea7dd02fbdea40eeeeee
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [6]:
import numpy as np
from pycuda.compiler import SourceModule
import pycuda.driver as cuda

matrix_size = 1024
h_matrix = np.random.randn(matrix_size, matrix_size).astype(np.float32)
h_result = np.empty_like(h_matrix)


mod = SourceModule("""
__global__ void matrix_square(float *dest, float *src, int N)
{

    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    // Ensure we are within matrix bounds
    if (x < N && y < N)
    {
        dest[x][y] = src[x][y] * 2 + 10;
    }
}
""")

matrix_square = mod.get_function("matrix_square")

matrix_square(
    cuda.Out(h_result),
    cuda.In(h_matrix),
    block=(1,1,1),
    grid=(1,1)
)

print(h_matrix)
print(h_result)


[[ 7.6013219e-01  1.0499647e+00 -9.5595884e-01 ...  1.8150159e+00
  -1.7314082e+00 -4.8224148e-01]
 [ 8.3277935e-01 -1.1902370e+00 -2.5119784e-01 ...  1.3711859e+00
  -1.1451401e+00  3.4103179e-01]
 [-4.2337549e-01  5.1245582e-01  1.6211729e+00 ...  1.3188177e+00
   3.0080092e-01  1.6934064e-03]
 ...
 [-1.4017274e+00 -9.4708443e-01  2.7156833e-01 ... -1.4746004e+00
  -1.0567567e+00  5.5754209e-01]
 [ 1.7680356e-01  3.1639042e-01 -1.2159512e+00 ...  4.9507465e-02
  -3.3341911e-01  9.9748179e-02]
 [ 1.2522911e+00  1.9796224e-01 -2.5866842e+00 ... -5.4036528e-01
  -1.5540253e+00  4.3695539e-02]]
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
